# Analysis — causal vs correlational features

**Analysis notebook.** This notebook only *imports* from `src/dna_interp/` and *reads artifacts* produced by `scripts/`. No heavy jobs or definitions live here.

Run a pipeline first, e.g. `python scripts/run_pipeline.py smoke=true`.

In [ ]:
import sys; sys.path.insert(0, '../src')
from pathlib import Path
import numpy as np, pandas as pd
from dna_interp.utils import load_config
from dna_interp import causal

cfg = load_config(['smoke=true'])           # match the run you want to inspect
ART = Path('..') / cfg.paths['artifacts']
L = int((ART / 'chosen_layer.txt').read_text().strip())
feats = pd.read_csv(ART / f'causal_features_layer{L}.csv')
ann = pd.read_csv(ART / f'features_layer{L}.csv')[['feature_id','label','top_motif']]
feats = feats.merge(ann, on='feature_id', how='left')
null_A = np.load(ART / f'null_A_layer{L}.npy')
thr = causal.null_threshold(null_A, cfg.causal.percentile)
print('layer', L, '| threshold', round(thr,3), '| causal fraction',
      round(causal.causal_fraction(feats.A_f.to_numpy(), thr),3))
feats.head()

## A_f by biological class (the per-class breakdown is the real story)

In [ ]:
g = feats.dropna(subset=['A_f']).groupby('label')['A_f'].agg(
    n='count', median='median', causal_frac=lambda x: (x>thr).mean())
g

## Headline figure: A_f distribution vs null

In [ ]:
import matplotlib.pyplot as plt
bins = np.linspace(-1,1,41)
plt.hist(null_A, bins=bins, density=True, alpha=.5, color='gray', label='random null')
plt.hist(feats.A_f.dropna(), bins=bins, density=True, alpha=.6, color='tab:blue', label='SAE features')
plt.axvline(thr, color='red', ls='--', label=f'95th pct null = {thr:.2f}')
plt.xlabel('A_f'); plt.ylabel('density'); plt.legend(); plt.show()

## Top causal features — spot-check biology
Inspect `top_motif` for the highest-A_f features; for the real run cross-reference JASPAR/cCRE.
Causal **and** `unannotated` features are candidate novel regulatory elements.

In [ ]:
feats.dropna(subset=['A_f']).sort_values('A_f', ascending=False).head(15)[
    ['feature_id','A_f','n_active_e','label','top_motif','p_value']]